# CS781 Phase 3 - Rhetorical Roles - notebook `cv-E-tail-w30tail`

**Cross-validation study on the TRAIN documents (rare labels).** The 80 training documents are split into 5 folds by
document (documents with Dissent / PrecedentOverruled are spread over the folds). Each run trains on 4 folds, picks its
epoch on the official dev set as before, and predicts the held-out fold (`oof_prob.npz`), dev and test.
Dev and test are never trained on. Models D and E, LoRA r16, full design. Variants:
`base` (reference), `w30` (H1: inverse-frequency class weights, cap 30), `tail` (H2: one extra end-of-document training
range per document and epoch), `w30tail` (H1 + H2). Analysis: CPU notebook `cs781-p3-cv-analysis`.

| ID | GPU | run | technique | change vs full design | question |
|---|---|---|---|---|---|
| Ecv_tail_f0 | GPU0 | `Ecv_tail_f0` | LoRA r16 | fold 0; +1 end-of-document range per doc/epoch | H2: see document endings more often |
| Ecv_tail_f2 | GPU0 | `Ecv_tail_f2` | LoRA r16 | fold 2; +1 end-of-document range per doc/epoch | H2: see document endings more often |
| Ecv_tail_f4 | GPU0 | `Ecv_tail_f4` | LoRA r16 | fold 4; +1 end-of-document range per doc/epoch | H2: see document endings more often |
| Ecv_w30tail_f1 | GPU0 | `Ecv_w30tail_f1` | LoRA r16 | fold 1; H1 + H2 | H1 + H2 together |
| Ecv_w30tail_f3 | GPU0 | `Ecv_w30tail_f3` | LoRA r16 | fold 3; H1 + H2 | H1 + H2 together |
| Ecv_tail_f1 | GPU1 | `Ecv_tail_f1` | LoRA r16 | fold 1; +1 end-of-document range per doc/epoch | H2: see document endings more often |
| Ecv_tail_f3 | GPU1 | `Ecv_tail_f3` | LoRA r16 | fold 3; +1 end-of-document range per doc/epoch | H2: see document endings more often |
| Ecv_w30tail_f0 | GPU1 | `Ecv_w30tail_f0` | LoRA r16 | fold 0; H1 + H2 | H1 + H2 together |
| Ecv_w30tail_f2 | GPU1 | `Ecv_w30tail_f2` | LoRA r16 | fold 2; H1 + H2 | H1 + H2 together |
| Ecv_w30tail_f4 | GPU1 | `Ecv_w30tail_f4` | LoRA r16 | fold 4; H1 + H2 | H1 + H2 together |

**Inputs:** dataset `cs781-p3-code` (latest version), `cs781-legal-data`, Phase-1 checkpoint of model E.
Internet ON, secret `HF_TOKEN`, GPU **T4 x2**. Save Version -> Save & Run All.
Expected: ~0.8 h after a 3-5 min smoke test. A run that fails the smoke test is dropped (its error is in the
report); the others continue.
**Outputs:** `submissions/<run>.zip` (ready to submit), `models/<run>.pt` (trained parameters + config + best epoch +
dev score), `bundle_cv-E-tail-w30tail.zip` (dev/test probabilities + reports, for the ensemble), `REPORT_cv-E-tail-w30tail.txt`.

In [1]:
import os, sys, json, re, glob, shutil, subprocess, time, threading, zipfile
W = "/kaggle/working"
WORK, P3DATA, RUNS, MODELS = f"{W}/p3code", f"{W}/p3data", f"{W}/runs", f"{W}/models"
CACHE = f"{W}/rr_cache.pkl"
for d in (RUNS, MODELS, f"{W}/logs", f"{W}/submissions"):
    os.makedirs(d, exist_ok=True)

def find_dir(name, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True), key=len)
    assert hits, f"{name} not found under {root}"
    return os.path.dirname(hits[0])

shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(find_dir("rr_train.py"), WORK)
os.chdir(WORK); sys.path.insert(0, WORK)
TOK_ROOT = find_dir("manifest.json")
print("code:", sorted(f for f in os.listdir(WORK) if f.endswith(".py")))

# ---- Phase-3 data from Hugging Face
from huggingface_hub import hf_hub_download, list_repo_files
from kaggle_secrets import UserSecretsClient
REPO = "Exploration-Lab/CS-781-Capstone"
tok_ = UserSecretsClient().get_secret("HF_TOKEN")
os.makedirs(P3DATA, exist_ok=True)
for f in list_repo_files(REPO, repo_type="dataset", token=tok_):
    if f.startswith("Phase_3") and f.endswith(".json"):
        shutil.copy(hf_hub_download(REPO, f, repo_type="dataset", token=tok_), os.path.join(P3DATA, os.path.basename(f)))
print("p3 data:", sorted(os.listdir(P3DATA)))

# dev truth files in the official evaluator's format (used to cross-check our metric)
import rr_data
for dom in ("CL", "IT"):
    json.dump({x["id"]: x["labels"] for x in rr_data.load_split(f"{P3DATA}/{dom}_dev.json")}, open(f"{W}/dev_truth_{dom}.json", "w"))

# ---- Phase-1 checkpoints (final.pt per model letter; continual-pretrained *_cpt preferred)
CKPT = {}
for p in sorted(glob.glob("/kaggle/input/**/final.pt", recursive=True)):
    m = re.search(r"run_([a-e])_[^/]*", p)
    if not m or "smoke" in p:
        continue
    k = m.group(1)
    if k not in CKPT or ("_cpt" in p and "_cpt" not in CKPT[k]):
        CKPT[k] = p
for k, v in sorted(CKPT.items()):
    print(f"  model {k}: {v}")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)

# ---- tokenize every split once (shared by all jobs) + CPU tests
t0 = time.time()
cache = rr_data.build_cache(P3DATA, TOK_ROOT, CACHE)
for k, v in cache["docs"].items():
    print(k, len(v), "docs", sum(len(x["sents"]) for x in v), "sentences")
print(f"cache built in {time.time() - t0:.0f}s")
r = subprocess.run("python tests_rr.py", shell=True, capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
assert r.returncode == 0, "unit tests failed"

code: ['adapt.py', 'attn_core.py', 'attn_cross.py', 'attn_gqa_local.py', 'attn_mha.py', 'attn_mla.py', 'evaluate_phase3.py', 'gated_deltanet.py', 'kv_cache.py', 'layers.py', 'models.py', 'ner_data.py', 'ner_model.py', 'p1_loader.py', 'peft.py', 'rr_data.py', 'rr_ensemble.py', 'rr_eval.py', 'rr_model.py', 'rr_train.py', 'tests_rr.py']


CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

p3 data: ['CL_dev.json', 'CL_test.json', 'CL_train.json', 'IT_dev.json', 'IT_test.json', 'IT_train.json', 'label_vocab.json']
  model e: /kaggle/input/models/mrinaldevnath/cs781-capstone-model-e/pytorch/phase-1/1/run_e_small_gpt_55m/final.pt
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

('CL', 'train') 40 docs 10593 sentences
('CL', 'dev') 5 docs 1396 sentences
('CL', 'test') 5 docs 1339 sentences
('IT', 'train') 40 docs 6252 sentences
('IT', 'dev') 5 docs 746 sentences
('IT', 'test') 5 docs 858 sentences
cache built in 3s
ok test_segments
ok test_pack_and_arrays
ok test_metric
ok test_files_and_validation
ok test_folds_and_tail
 


In [2]:
def cmd(model, out, extra, model_dir):
    return (f"python rr_train.py --ckpt {CKPT[model]} --tok_root {TOK_ROOT} --data_dir {P3DATA} --cache {CACHE} "
            f"--out {out} --model_dir {model_dir} {extra}")

def run(tag, c, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    with open(f"{W}/logs/{tag}.txt", "w") as f:
        p = subprocess.run(c, shell=True, cwd=WORK, env=env, stdout=f, stderr=subprocess.STDOUT)
    tail = open(f"{W}/logs/{tag}.txt").read().strip().splitlines()[-2:]
    print(f"[GPU{gpu}] {tag}: exit {p.returncode} | " + " / ".join(tail)[-300:], flush=True)
    return p.returncode

def run_queues(queues, smoke=False):
    """queues: one list of experiments per GPU, run in parallel. smoke: 1 tiny epoch per experiment."""
    status = {}
    def worker(gpu, jobs):
        for e in jobs:
            t = ("smoke_" if smoke else "") + e["tag"]
            extra = e["args"] + (" --max_docs 1 --max_sents 40 --epochs 1" if smoke else "")
            if smoke and e is not jobs[0]:
                extra += " --no_test"
            status[t] = run(t, cmd(e["model"], f"{RUNS}/{t}", extra, f"{W}/smoke_models" if smoke else MODELS), gpu)
    th = [threading.Thread(target=worker, args=(g, q)) for g, q in enumerate(queues) if q]
    t0 = time.time()
    for t in th: t.start()
    for t in th: t.join()
    print(f"{'smoke' if smoke else 'runs'} done in {(time.time() - t0) / 60:.1f} min:", status)
    return status

def smoke(queues):
    """Runs 1 tiny epoch of every experiment. A failing experiment is dropped (its error is printed and
    reported) instead of stopping the notebook; the notebook stops only if every experiment fails."""
    st = run_queues(queues, smoke=True)
    bad = [k for k, v in st.items() if v != 0]
    for k in bad:
        print(f"===== SMOKE FAILED, dropped: {k}\n" + open(f"{W}/logs/{k}.txt").read()[-3000:])
    assert len(bad) < len(st), "every experiment failed the smoke test"
    for k in st:
        shutil.rmtree(f"{RUNS}/{k}", ignore_errors=True)
    shutil.rmtree(f"{W}/smoke_models", ignore_errors=True)
    print("SMOKE OK" if not bad else f"SMOKE OK except {len(bad)} dropped: {bad}")
    return [[e for e in q if "smoke_" + e["tag"] not in bad] for q in queues]

## Experiments (one queue per GPU)

In [3]:
NAME = 'cv-E-tail-w30tail'
QUEUES = [[{'id': 'Ecvtail0',
   'tag': 'Ecv_tail_f0',
   'model': 'e',
   'tech': 'LoRA r16',
   'change': 'fold 0; +1 end-of-document range per doc/epoch',
   'args': '--method lora --rank 16 --alpha 32 --tail_ranges 1 --n_folds 5 --fold 0'},
  {'id': 'Ecvtail2',
   'tag': 'Ecv_tail_f2',
   'model': 'e',
   'tech': 'LoRA r16',
   'change': 'fold 2; +1 end-of-document range per doc/epoch',
   'args': '--method lora --rank 16 --alpha 32 --tail_ranges 1 --n_folds 5 --fold 2'},
  {'id': 'Ecvtail4',
   'tag': 'Ecv_tail_f4',
   'model': 'e',
   'tech': 'LoRA r16',
   'change': 'fold 4; +1 end-of-document range per doc/epoch',
   'args': '--method lora --rank 16 --alpha 32 --tail_ranges 1 --n_folds 5 --fold 4'},
  {'id': 'Ecvw30t1',
   'tag': 'Ecv_w30tail_f1',
   'model': 'e',
   'tech': 'LoRA r16',
   'change': 'fold 1; H1 + H2',
   'args': '--method lora --rank 16 --alpha 32 --class_weight inv --weight_cap 30 --tail_ranges 1 --n_folds 5 --fold 1'},
  {'id': 'Ecvw30t3',
   'tag': 'Ecv_w30tail_f3',
   'model': 'e',
   'tech': 'LoRA r16',
   'change': 'fold 3; H1 + H2',
   'args': '--method lora --rank 16 --alpha 32 --class_weight inv --weight_cap 30 --tail_ranges 1 --n_folds 5 --fold 3'}],
 [{'id': 'Ecvtail1',
   'tag': 'Ecv_tail_f1',
   'model': 'e',
   'tech': 'LoRA r16',
   'change': 'fold 1; +1 end-of-document range per doc/epoch',
   'args': '--method lora --rank 16 --alpha 32 --tail_ranges 1 --n_folds 5 --fold 1'},
  {'id': 'Ecvtail3',
   'tag': 'Ecv_tail_f3',
   'model': 'e',
   'tech': 'LoRA r16',
   'change': 'fold 3; +1 end-of-document range per doc/epoch',
   'args': '--method lora --rank 16 --alpha 32 --tail_ranges 1 --n_folds 5 --fold 3'},
  {'id': 'Ecvw30t0',
   'tag': 'Ecv_w30tail_f0',
   'model': 'e',
   'tech': 'LoRA r16',
   'change': 'fold 0; H1 + H2',
   'args': '--method lora --rank 16 --alpha 32 --class_weight inv --weight_cap 30 --tail_ranges 1 --n_folds 5 --fold 0'},
  {'id': 'Ecvw30t2',
   'tag': 'Ecv_w30tail_f2',
   'model': 'e',
   'tech': 'LoRA r16',
   'change': 'fold 2; H1 + H2',
   'args': '--method lora --rank 16 --alpha 32 --class_weight inv --weight_cap 30 --tail_ranges 1 --n_folds 5 --fold 2'},
  {'id': 'Ecvw30t4',
   'tag': 'Ecv_w30tail_f4',
   'model': 'e',
   'tech': 'LoRA r16',
   'change': 'fold 4; H1 + H2',
   'args': '--method lora --rank 16 --alpha 32 --class_weight inv --weight_cap 30 --tail_ranges 1 --n_folds 5 --fold 4'}]]
ALL_QUEUES = [list(q) for q in QUEUES]                    # the full plan (the report lists dropped runs too)

## 1. Smoke test

In [4]:
QUEUES = smoke(QUEUES)

[GPU0] smoke_Ecv_tail_f0: exit 0 | on.zip  {'Fact': 1929, 'Issue': 0, 'ArgumentPetitioner': 4, 'ArgumentRespondent': 2, 'Statute': 0, 'Dissent': 0, 'PrecedentReliedUpon': 0, 'PrecedentNotReliedUpon': 0, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 0, 'RatioOfTheDecision': 262, 'RulingByPresentCourt': 0, 'None': 0} / [11:06:15] DONE
[GPU1] smoke_Ecv_tail_f1: exit 0 | on.zip  {'Fact': 1929, 'Issue': 0, 'ArgumentPetitioner': 4, 'ArgumentRespondent': 2, 'Statute': 0, 'Dissent': 0, 'PrecedentReliedUpon': 0, 'PrecedentNotReliedUpon': 0, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 0, 'RatioOfTheDecision': 262, 'RulingByPresentCourt': 0, 'None': 0} / [11:06:15] DONE
[GPU1] smoke_Ecv_tail_f3: exit 0 |   None                     F1 0.000  P 0.000  R 0.000  gold     0  pred     0 / [11:06:26] DONE
[GPU0] smoke_Ecv_tail_f2: exit 0 |   RulingByPresentCourt     F1 0.000  P 0.000  R 0.000  gold     0  pred     0 / [11:06:26] DONE
[GPU0] smoke_Ecv_tail_f4: exit 0 |   None                     F1 0

## 2. Runs

In [5]:
status = run_queues(QUEUES)

[GPU0] Ecv_tail_f0: exit 0 | t': 367, 'Issue': 14, 'ArgumentPetitioner': 160, 'ArgumentRespondent': 152, 'Statute': 162, 'Dissent': 0, 'PrecedentReliedUpon': 282, 'PrecedentNotReliedUpon': 46, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 101, 'RatioOfTheDecision': 849, 'RulingByPresentCourt': 39, 'None': 25} / [11:14:45] DONE
[GPU1] Ecv_tail_f1: exit 0 | t': 416, 'Issue': 13, 'ArgumentPetitioner': 169, 'ArgumentRespondent': 142, 'Statute': 164, 'Dissent': 0, 'PrecedentReliedUpon': 325, 'PrecedentNotReliedUpon': 22, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 129, 'RatioOfTheDecision': 751, 'RulingByPresentCourt': 38, 'None': 28} / [11:17:12] DONE
[GPU0] Ecv_tail_f2: exit 0 | ': 398, 'Issue': 14, 'ArgumentPetitioner': 113, 'ArgumentRespondent': 205, 'Statute': 144, 'Dissent': 15, 'PrecedentReliedUpon': 349, 'PrecedentNotReliedUpon': 31, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 106, 'RatioOfTheDecision': 759, 'RulingByPresentCourt': 39, 'None': 24} / [11:23:01] DONE
[GPU1] Ecv

## 3. Report

In [6]:
SHORT = {"ArgumentPetitioner": "ArgPet", "ArgumentRespondent": "ArgResp", "PrecedentReliedUpon": "PrecRel",
         "PrecedentNotReliedUpon": "PrecNRel", "PrecedentOverruled": "PrecOvr", "RulingByLowerCourt": "RulLower",
         "RatioOfTheDecision": "Ratio", "RulingByPresentCourt": "RulPres"}

def official(tag):
    """Official evaluator on this run's dev predictions (cross-check of our metric)."""
    r = subprocess.run(f"python evaluate_phase3.py --truth-cl {W}/dev_truth_CL.json --truth-it {W}/dev_truth_IT.json "
                       f"--prediction {RUNS}/{tag}/dev_pred.json --labels {P3DATA}/label_vocab.json",
                       shell=True, capture_output=True, text=True, cwd=WORK)
    m = re.search(r"Macro-F1:\s+([0-9.]+)", r.stdout)
    return float(m.group(1)) if m else float("nan")

def log_tail(name, n=2):
    p = f"{W}/logs/{name}.txt"
    return " / ".join(open(p).read().strip().splitlines()[-n:])[-250:] if os.path.exists(p) else ""

def summarize(name, queues):
    exps = [e for q in queues for e in q]
    L = [f"=== REPORT {name} ===",
         "ID  | run                          | technique          | change                     | dev F1 | official | CL     | IT     | P      | R      | ep | trainable | min  | tok/s  | mem"]
    rows = []
    for e in exps:
        p = f"{RUNS}/{e['tag']}/summary.json"
        if not os.path.exists(p):
            why = "SMOKE FAILED (dropped)" if not os.path.exists(f"{W}/logs/{e['tag']}.txt") else "FAILED"
            L.append(f"{e['id']} | {e['tag']:28s} | {why}: {log_tail(e['tag']) or log_tail('smoke_' + e['tag'])}")
            continue
        s = json.load(open(p)); dv = s["dev"]
        off = official(e["tag"])
        rows.append((e, s))
        L.append(f"{e['id']} | {e['tag']:28s} | {e['tech'][:18]:18s} | {e['change'][:26]:26s} | {dv['macro_f1']:.4f} | {off:.4f}   | "
                 f"{dv['macro_f1_CL']:.4f} | {dv['macro_f1_IT']:.4f} | {dv['macro_p']:.4f} | {dv['macro_r']:.4f} | {s['best_epoch']:2d} | "
                 f"{s['params_trainable'] / 1e6:8.2f}M | {s['train_minutes']:4.0f} | {s.get('tok_s') or 0:6,.0f} | {s.get('peak_mem_gb') or 0:.1f}G")
        shutil.copy(f"{RUNS}/{e['tag']}/submission.zip", f"{W}/submissions/{e['tag']}.zip")
    if rows:
        labs = rows[0][1]["labels"]
        L.append("\nper-label dev F1 (official: all 13 labels count; Dissent and PrecedentOverruled have no dev sentences -> 0)")
        L.append(f"{'ID':4s} " + " ".join(f"{SHORT.get(l, l)[:8]:>8s}" for l in labs))
        for e, s in rows:
            pl = s["dev"]["per_label"]
            L.append(f"{e['id']:4s} " + " ".join(f"{pl[l]['f1']:8.2f}" for l in labs))
        L.append("\ntop confusions (gold->pred:count)")
        for e, s in rows:
            L.append(f"{e['id']}: " + " ".join(s["dev"]["top_confusions"][:6]))
    bundle = f"{W}/bundle_{name}.zip"                     # everything except model weights (those stay in models/)
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
        for e in exps:
            d = f"{RUNS}/{e['tag']}"
            if os.path.isdir(d):
                for f in os.listdir(d):
                    z.write(f"{d}/{f}", arcname=f"{e['tag']}/{f}")
    L.append(f"\nbundle: {bundle} ({os.path.getsize(bundle) / 2**20:.1f} MB)")
    L.append("models: " + ", ".join(f"{f} ({os.path.getsize(f'{MODELS}/{f}') / 2**20:.0f} MB)" for f in sorted(os.listdir(MODELS))))
    L.append("submission files: " + ", ".join(sorted(os.listdir(f"{W}/submissions"))))
    rep = "\n".join(L)
    open(f"{W}/REPORT_{name}.txt", "w").write(rep)
    print(rep + "\n=== END REPORT ===")

summarize(NAME, ALL_QUEUES)

=== REPORT cv-E-tail-w30tail ===
ID  | run                          | technique          | change                     | dev F1 | official | CL     | IT     | P      | R      | ep | trainable | min  | tok/s  | mem
Ecvtail0 | Ecv_tail_f0                  | LoRA r16           | fold 0; +1 end-of-document | 0.5074 | 0.5074   | 0.4310 | 0.5069 | 0.5484 | 0.5027 |  4 |     3.42M |    8 | 12,955 | 2.3G
Ecvtail2 | Ecv_tail_f2                  | LoRA r16           | fold 2; +1 end-of-document | 0.5141 | 0.5141   | 0.4408 | 0.5132 | 0.5330 | 0.5049 |  5 |     3.42M |    8 | 13,020 | 2.3G
Ecvtail4 | Ecv_tail_f4                  | LoRA r16           | fold 4; +1 end-of-document | 0.4982 | 0.4982   | 0.4301 | 0.5046 | 0.5519 | 0.4775 |  5 |     3.42M |    9 | 12,885 | 2.3G
Ecvw30t1 | Ecv_w30tail_f1               | LoRA r16           | fold 1; H1 + H2            | 0.5067 | 0.5067   | 0.4769 | 0.4941 | 0.5135 | 0.5352 |  3 |     3.42M |    7 | 12,909 | 2.3G
Ecvw30t3 | Ecv_w30tail_f3               | L